In [11]:
import pandas as pd

df = pd.read_csv("../data/marketing_AB.csv")
print(df.shape)
df.head()

(588101, 7)


,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [12]:
df = df.drop(columns="Unnamed: 0")
df.columns = ["user_id", "group", "converted", "total_ads", "most_ads_day", "most_ads_hour"]
df.dtypes

user_id           int64
group            object
converted          bool
total_ads         int64
most_ads_day     object
most_ads_hour     int64
dtype: object

In [17]:
duplicates = df["user_id"].duplicated().sum()
missing = df.isna().sum()
unique_groups = df["group"].unique()
unique_converted = df["converted"].unique()

print("Количество дублей user_id:", duplicates)
print("Количество пропусков по столбцам:\n", missing, sep = "")
print("Уникальные значения в столбце group:", unique_groups)
print("Уникальные значения в столбце converted:", unique_converted)

Количество дублей user_id: 0
Количество пропусков по столбцам:
user_id          0
group            0
converted        0
total_ads        0
most_ads_day     0
most_ads_hour    0
dtype: int64
Уникальные значения в столбце group: ['ad' 'psa']
Уникальные значения в столбце converted: [False  True]


In [29]:
import sqlite3

con = sqlite3.connect(":memory:")
df.to_sql("users", con, index = False)

query = """
SELECT
    "group",
    users,
    users * 1.0 / SUM(users) OVER () AS shares,
    conversions,
    conv_rate
FROM (
    SELECT
        "group",
        COUNT(*) AS users,
        SUM(converted) AS conversions,
        AVG(converted) AS conv_rate
    FROM users
    GROUP BY "group"
    )
"""
pd.read_sql(query, con)

,group,users,shares,conversions,conv_rate
0,ad,564577,0.96,14423,0.025547
1,psa,23524,0.04,420,0.017854


In [35]:
from scipy import stats

n_ad = (df["group"] == "ad").sum()
n_ad = (df["group"] == "psa").sum()
n_total = n_ad + n_psa

observed = [n_ad, n_psa]
expected = [n_total * 0.96, n_total * 0.04]

chi2, p_value = stats.chisquare(observed, f_exp=expected)

print ("Наблюдаем:", observed)
print ("Ожидали:  ", expected)
print ("p_value:  ", p_value)

Наблюдаем: [np.int64(23524), 23524]
Ожидали:   [np.float64(45166.08), np.float64(1881.92)]
p_value:   0.0


In [34]:
counts = df["group"].value_counts()
counts

group
ad     564577
psa     23524
Name: count, dtype: int64